<a href="https://colab.research.google.com/github/mharrell/TerrariaGuide/blob/main/terraria_train_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TerrariaGuide - 8B Fine-Tune
Run each cell in order. Make sure **Runtime > Change runtime type** is set to **T4 GPU** before starting.

In [1]:
# Cell 1 - Install dependencies
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!pip install unsloth trl transformers datasets accelerate bitsandbytes peft sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [2]:
# Cell 2 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
TRAINING_FILE = '/content/drive/MyDrive/TerrariaGuide/terraria_training_pairs.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/TerrariaGuide/models/terraria-guide-8b'

if os.path.exists(TRAINING_FILE):
    print(f'Training file found!')
    print(f'Lines: {sum(1 for _ in open(TRAINING_FILE))}')
else:
    print(f'ERROR: Training file not found at {TRAINING_FILE}')
    print('Please upload terraria_training_pairs.jsonl to your Drive at TerrariaGuide/terraria_training_pairs.jsonl')

Mounted at /content/drive
Training file found!
Lines: 32923


In [3]:
# Cell 3 - Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

GPU: Tesla T4
VRAM: 14.6 GB
CUDA: 12.8


In [4]:
# Cell 4 - Load model
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

print('Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model ready!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model...
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready!


In [5]:
# Cell 5 - Load and format dataset
from datasets import load_dataset

def format_prompt(example):
    messages = [
        {
            'role': 'system',
            'content': 'You are TerrariaGuide, an expert assistant for the game Terraria. Answer questions accurately about items, crafting, bosses, enemies, biomes, and game mechanics.'
        },
        {'role': 'user',      'content': example['instruction']},
        {'role': 'assistant', 'content': example['output']},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

def is_within_seq_length(example):
    token_count = len(tokenizer(example['text'], add_special_tokens=False)['input_ids'])
    return token_count <= MAX_SEQ_LENGTH

print('Loading dataset...')
dataset = load_dataset('json', data_files=TRAINING_FILE, split='train')
dataset = dataset.map(format_prompt)
before = len(dataset)
dataset = dataset.filter(is_within_seq_length)
print(f'Dataset size: {len(dataset)} pairs ({before - len(dataset)} filtered for exceeding {MAX_SEQ_LENGTH} tokens)')

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/32923 [00:00<?, ? examples/s]

Filter:   0%|          | 0/32923 [00:00<?, ? examples/s]

Dataset size: 32880 pairs (43 filtered for exceeding 1024 tokens)


In [ ]:
# Cell 6 - Train
from trl import SFTTrainer
from transformers import TrainingArguments

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=training_args,
)

print('Starting training...')
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/32880 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,880 | Num Epochs = 1 | Total steps = 4,110
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
25,2.039175
50,0.942534
75,0.847740
100,0.798193
125,0.791940
150,0.756370
175,0.737235
200,0.768798
225,0.784935
250,0.735834


Step,Training Loss
25,2.039175
50,0.942534
75,0.847740
100,0.798193
125,0.791940
150,0.756370
175,0.737235
200,0.768798
225,0.784935
250,0.735834


In [ ]:
# Cell 7 - Save model to Drive
print('Saving model to Google Drive...')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

In [ ]:
# Cell 8 - Quick test
FastLanguageModel.for_inference(model)

def ask(question):
    messages = [
        {
            'role': 'system',
            'content': 'You are TerrariaGuide, an expert assistant for the game Terraria. Answer questions accurately about items, crafting, bosses, enemies, biomes, and game mechanics.'
        },
        {'role': 'user', 'content': question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('Test 1:', ask('What does the Eye of Cthulhu drop in Terraria?'))
print()
print('Test 2:', ask('How do I craft a Molten Pickaxe in Terraria?'))
print()
print('Test 3:', ask('What is the Wall of Flesh in Terraria?'))